# Avance 2 – Integración multimodal del asistente
**Proyecto:** Asistente de reclutamiento del restaurante El Sauce
**Alumna:** Juana Paola Puente Huerta

Este cuaderno une los dos componentes del Avance 2 con el chatbot del Avance 1:

| Componente | Qué hace en el chatbot |
|---|---|
| **Textual (NLP)** | Un SVM con TF-IDF identifica la intención del mensaje, aunque tenga faltas de ortografía. |
| **Visual (CNN)** | EfficientNetB0 clasifica la foto o PDF que envía el candidato: INE, licencia, comprobante u otro. |
| **Integración** | Al pasar el prefiltro, el bot pide los documentos, interpreta el resultado del modelo visual y se lo comunica al candidato antes de agendar la entrevista. |

**Antes de ejecutar**, en tu Drive deben estar:
```
ChatbotRH/
   base_conocimiento.xlsx
   dataset_intenciones.csv
   modelo/modelo_documentos.keras     <- lo guardó el cuaderno del clasificador de documentos
   modelo/clases.json
   pruebas/                           <- fotos NUEVAS para probar (no usadas en el entrenamiento)
```

## 1. Configuración

In [ ]:
!pip install -q pillow-heif pymupdf

import os, re, json, unicodedata, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
from pillow_heif import register_heif_opener
import fitz
import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

register_heif_opener()
nltk.download('stopwords', quiet=True)

from google.colab import drive, files
drive.mount('/content/drive')

RUTA         = '/content/drive/MyDrive/ChatbotRH'
RUTA_MODELO  = os.path.join(RUTA, 'modelo')
RUTA_PRUEBAS = os.path.join(RUTA, 'pruebas')
MOSTRAR_ANALISIS = True    # muestra la intención y la clase detectadas (evidencia del procesamiento)

## 2. Base de conocimiento
Igual que en el Avance 1: la información del restaurante se lee desde el Excel.

In [ ]:
def sin_acentos(texto):
    texto = unicodedata.normalize('NFD', str(texto).lower())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn').strip()

def es_si(valor):
    return sin_acentos(valor) in ('si', 's', 'yes', 'true', '1')

def cargar_base(ruta):
    puestos_df = pd.read_excel(ruta, sheet_name='Puestos').dropna(subset=['Puesto'])
    general_df = pd.read_excel(ruta, sheet_name='General')
    entrevista_df = pd.read_excel(ruta, sheet_name='Entrevista')
    puestos, sinonimos = {}, {}
    for _, fila in puestos_df.iterrows():
        palabras = [sin_acentos(p) for p in str(fila['Palabras clave']).split(',') if p.strip()]
        clave = palabras[0] if palabras else sin_acentos(fila['Puesto'])
        puestos[clave] = {
            'nombre': str(fila['Puesto']).strip(),
            'horarios': [h.strip() for h in str(fila['Horarios']).split('|') if h.strip()],
            'sueldo': str(fila['Sueldo']).strip(),
            'experiencia': int(fila['Experiencia (años)']) if pd.notna(fila['Experiencia (años)']) else 0,
            'propinas': es_si(fila['Propinas']),
            'requiere_licencia': es_si(fila['Requiere licencia']),
            'funciones': str(fila['Funciones']).strip(),
        }
        sinonimos[clave] = palabras
    general = dict(zip(general_df['Concepto'].str.strip(), general_df['Valor']))
    general['edad_minima'] = int(general.get('edad_minima', 18))
    entrevista = dict(zip(entrevista_df['Concepto'].str.strip(), entrevista_df['Valor'].astype(str)))
    return puestos, sinonimos, general, entrevista

PUESTOS, SINONIMOS_PUESTO, GENERAL, ENTREVISTA = cargar_base(os.path.join(RUTA, 'base_conocimiento.xlsx'))
print(f"Base de conocimiento: {len(PUESTOS)} puestos")

## 3. Componente textual: clasificador de intenciones
Se usa el mismo preprocesamiento y el mismo modelo del cuaderno de intenciones
(SVM con TF-IDF de palabras y fragmentos de letras). Se entrena al iniciar, porque tarda menos de un segundo.

In [ ]:
stemmer = SnowballStemmer('spanish')

def normalizar(texto):
    texto = sin_acentos(texto)
    texto = re.sub(r'[^a-z0-9 ]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

CONSERVAR = {'cuanto', 'cuanta', 'cuantos', 'cuantas', 'donde', 'cuando', 'como', 'cual', 'cuales', 'sin', 'no'}
STOPWORDS = {normalizar(p) for p in stopwords.words('spanish')} - CONSERVAR

def preprocesar(texto):
    """Normalización, tokenización, eliminación de stopwords y stemming."""
    tokens = wordpunct_tokenize(normalizar(texto))
    raices = [stemmer.stem(t) for t in tokens if t not in STOPWORDS]
    return ' '.join(raices) if raices else normalizar(texto)

datos = pd.read_csv(os.path.join(RUTA, 'dataset_intenciones.csv'))
clasificador_texto = Pipeline([
    ('tfidf', FeatureUnion([
        ('palabras', TfidfVectorizer(ngram_range=(1, 2))),
        ('letras',   TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))),
    ])),
    ('clf', CalibratedClassifierCV(LinearSVC(C=1.0), cv=3)),
])
clasificador_texto.fit(datos['texto'].apply(preprocesar), datos['intencion'])
UMBRAL_TEXTO = 0.30

def clasificar_intencion(mensaje):
    p = clasificador_texto.predict_proba([preprocesar(mensaje)])[0]
    i = int(p.argmax())
    intencion = clasificador_texto.classes_[i] if p[i] >= UMBRAL_TEXTO else 'no_entendido'
    return intencion, float(p[i])

print("Clasificador de intenciones entrenado con", len(datos), "frases")
print(clasificar_intencion('cuanto pagan d mesra'))

## 4. Componente visual: clasificador de documentos
Se carga el modelo EfficientNetB0 que se entrenó y guardó en Drive. Si la confianza es menor al **60 %**,
el resultado se trata como **dudoso** y el chatbot pide una foto más clara.

In [ ]:
import keras
modelo_vision = keras.models.load_model(os.path.join(RUTA_MODELO, 'modelo_documentos.keras'))
CLASES_DOC = json.load(open(os.path.join(RUTA_MODELO, 'clases.json')))
UMBRAL_IMAGEN = 0.60
TAM = 224

def cargar_imagen(ruta, tam=TAM):
    if ruta.lower().endswith('.pdf'):
        pix = fitz.open(ruta)[0].get_pixmap(dpi=100)
        img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
    else:
        img = ImageOps.exif_transpose(Image.open(ruta)).convert('RGB')
    return ImageOps.pad(img, (tam, tam), color=(0, 0, 0))

def clasificar_documento(ruta):
    """Regresa (clase, confianza). Con confianza baja, la clase es 'dudoso'."""
    x = np.array(cargar_imagen(ruta), dtype=np.float32)[None, ...]
    p = modelo_vision.predict(x, verbose=0)[0]
    clase, confianza = CLASES_DOC[int(p.argmax())], float(p.max())
    return ('dudoso' if confianza < UMBRAL_IMAGEN else clase), confianza

def mostrar_documento(ruta, clase):
    """Muestra una miniatura del archivo recibido (difuminada si es identificación)."""
    img = cargar_imagen(ruta, 300)
    if clase in ('ine', 'licencia', 'dudoso'):
        img = img.filter(ImageFilter.GaussianBlur(6))
    plt.figure(figsize=(2.4, 2.4)); plt.imshow(img); plt.axis('off'); plt.show()

print("Modelo visual cargado. Clases:", CLASES_DOC)

## 5. Chatbot integrado
Cambios respecto al Avance 1:
- La intención se identifica con el **clasificador SVM** en lugar de palabras clave exactas.
- El puesto se reconoce aunque esté mal escrito («mesra» → mesero), comparando con los sinónimos del Excel.
- Nuevas respuestas para `consulta_rh` y `fuera_de_tema`.
- **Nuevo paso de documentos:** si el candidato es compatible, se le piden INE y comprobante de domicilio
  (y licencia si es repartidor). El modelo visual clasifica cada archivo y el chatbot comunica el resultado.

In [ ]:
# ---------- Utilidades del flujo ----------
def tokens_de(mensaje):
    return normalizar(mensaje).split()

def contiene(texto, frase):
    return re.search(r'\b' + re.escape(frase) + r'\b', texto) is not None

def identificar_puesto(tokens):
    for clave, sinonimos in SINONIMOS_PUESTO.items():          # 1) coincidencia exacta
        if any(t in sinonimos for t in tokens):
            return clave
    todos = {s: c for c, lista in SINONIMOS_PUESTO.items() for s in lista}
    for t in tokens:                                            # 2) palabra mal escrita
        if len(t) >= 5:
            parecida = difflib.get_close_matches(t, list(todos), n=1, cutoff=0.85)
            if parecida:
                return todos[parecida[0]]
    return None

SI = ['si', 'claro', 'sip', 'simon', 'correcto', 'por supuesto', 'ok', 'va']
NO = ['no', 'nop', 'nel', 'negativo']

def respuesta_si_no(texto):
    if any(contiene(texto, n) for n in NO): return False
    if any(contiene(texto, s) for s in SI): return True
    return None

def extraer_numero(tokens):
    for t in tokens:
        if t.isdigit(): return int(t)
    palabras = {'cero': 0, 'ninguno': 0, 'nada': 0, 'uno': 1, 'un': 1, 'dos': 2, 'tres': 3,
                'cuatro': 4, 'cinco': 5, 'seis': 6, 'diez': 10}
    for t in tokens:
        if t in palabras: return palabras[t]
    return None

def lista_puestos():
    return '\n'.join(f"  • {p['nombre']}" for p in PUESTOS.values())

def puestos_sin_experiencia():
    return ', '.join(p['nombre'] for p in PUESTOS.values() if p['experiencia'] == 0)

def texto_experiencia(p):
    if p['experiencia'] == 0: return 'no se requiere experiencia'
    return f"se requiere mínimo {p['experiencia']} año{'s' if p['experiencia'] > 1 else ''} de experiencia"

def tabla_resumen(campo):
    if campo == 'sueldo':
        return '\n'.join(f"  • {p['nombre']}: {p['sueldo']}" for p in PUESTOS.values())
    return '\n'.join(f"  • {p['nombre']}: {' / '.join(p['horarios'])}" for p in PUESTOS.values())

MENU = ("Puedo ayudarte con: vacantes, funciones, sueldos, horarios, experiencia, prestaciones, "
        "propinas, documentos y entrevistas.\nSi quieres postularte, escribe \"me interesa\".")

In [ ]:
# ---------- Respuestas a preguntas frecuentes ----------
def responder_faq(intencion, puesto):
    p = PUESTOS.get(puesto)
    if intencion == 'saludo':
        return "¡Hola! Soy el asistente de reclutamiento de El Sauce. 😊\n" + MENU
    if intencion == 'vacantes':
        return ("Estos son los puestos que estamos contratando:\n" + lista_puestos() +
                "\n¿Te interesa alguno? Pregúntame por sus funciones, sueldo u horario.")
    if intencion == 'funciones':
        return f"El puesto de {p['nombre']} se encarga de {p['funciones']}" if p else \
               "¿De qué puesto quieres conocer las funciones?\n" + lista_puestos()
    if intencion == 'sueldo':
        if p:
            return f"El sueldo de {p['nombre']} es de {p['sueldo']}{' más propinas' if p['propinas'] else ''}."
        return "Estos son los sueldos por puesto:\n" + tabla_resumen('sueldo')
    if intencion == 'horarios':
        if p:
            return f"Horarios de {p['nombre']}: {' / '.join(p['horarios'])}. {GENERAL['turnos']}"
        return "Horarios por puesto:\n" + tabla_resumen('horarios') + "\n" + GENERAL['turnos']
    if intencion == 'flexibilidad':
        return GENERAL['flexibilidad'] + " Puedes comentarlo en tu entrevista."
    if intencion == 'experiencia':
        if p: return f"Para {p['nombre']} {texto_experiencia(p)}."
        return f"Algunos puestos piden experiencia y otros no. Puedes entrar sin experiencia como: {puestos_sin_experiencia()}."
    if intencion == 'prestaciones':
        return f"Ofrecemos:\n  • {GENERAL['descanso']}\n  • {GENERAL['comida']}\n  • Propinas (según el puesto)"
    if intencion == 'propinas':
        if p: return f"El puesto de {p['nombre']} {'sí' if p['propinas'] else 'no'} recibe propinas."
        return "Sí se reparten propinas entre: " + ', '.join(x['nombre'] for x in PUESTOS.values() if x['propinas']) + "."
    if intencion == 'documentos':
        con_lic = [x['nombre'] for x in PUESTOS.values() if x['requiere_licencia']]
        return (f"Los documentos obligatorios son: {GENERAL['documentos']}. Es deseable una {GENERAL['deseable']}. "
                f"Para {', '.join(con_lic)} también se requiere licencia de conducir vigente. "
                "Puedes enviarlos por aquí al completar el prefiltro (escribe \"me interesa\").")
    if intencion == 'entrevista':
        return (f"Las entrevistas son {ENTREVISTA['modalidad']}es en {ENTREVISTA['lugar']}, "
                f"de {ENTREVISTA['dias']} de {ENTREVISTA['horario']}, y duran {ENTREVISTA['duracion']}. "
                "Para agendar primero completa el prefiltro (escribe \"me interesa\").")
    if intencion == 'consulta_rh':
        return ("Esa información te la explica directamente el área de recursos humanos durante la entrevista. 🙂\n"
                "¿Te puedo ayudar con algo más sobre las vacantes?")
    if intencion == 'fuera_de_tema':
        return ("Este chat es solo para reclutamiento de personal de El Sauce. Para pedidos, reservaciones o "
                "dudas del menú, comunícate directamente con el restaurante.\n" + MENU)
    if intencion == 'despedida':
        return "¡Gracias por tu interés en El Sauce! Mucho éxito. 👋"
    return "Perdón, no entendí tu mensaje. 🤔\n" + MENU

In [ ]:
# ---------- Agenda de entrevistas (simulada en memoria) ----------
DIAS_HABILES = ['lunes', 'martes', 'miercoles', 'jueves', 'viernes']
AGENDA = {d: {} for d in DIAS_HABILES}

def a_min(h):
    hh, mm = h.split(':'); return int(hh) * 60 + int(mm)

def a_txt(m):
    return f"{m // 60}:{m % 60:02d}"

def espacios_del_dia():
    inicio, fin = re.findall(r'\d{1,2}:\d{2}', ENTREVISTA['horario'])[:2]
    dur = int(re.search(r'\d+', ENTREVISTA['duracion']).group())
    return [a_txt(m) for m in range(a_min(inicio), a_min(fin) - dur + 1, dur)]

def espacios_libres(dia):
    return [h for h in espacios_del_dia() if h not in AGENDA[dia]]

def extraer_dia(texto):
    for d in DIAS_HABILES:
        if contiene(texto, d): return d
    if contiene(texto, 'sabado') or contiene(texto, 'domingo'): return 'fin_de_semana'
    return None

def extraer_hora(texto):
    m = re.search(r'\b(\d{1,2})(?:\s+(\d{2}))?\b', texto)
    if not m: return None
    h, mins = int(m.group(1)), int(m.group(2) or 0)
    if 1 <= h <= 7: h += 12
    return a_txt(h * 60 + mins)

def ver_agenda():
    citas = [(d, h, AGENDA[d][h]) for d in DIAS_HABILES for h in sorted(AGENDA[d], key=a_min)]
    for d, h, n in citas: print(f"{d.capitalize():<10} {h:>5}  {n}")
    if not citas: print("No hay entrevistas agendadas.")

In [ ]:
# ---------- Paso de documentos: integración con el modelo visual ----------
NOMBRE_DOC = {'ine': 'INE', 'licencia': 'licencia de conducir',
              'comprobante': 'comprobante de domicilio (recibo de luz o agua)'}
MAX_INTENTOS = 3

def pedir_documento(estado):
    doc = estado['docs_pendientes'][0]
    return f"📎 Envía una foto o PDF de tu {NOMBRE_DOC[doc]}."

def procesar_documento(estado, ruta):
    esperado = estado['docs_pendientes'][0]
    clase, confianza = clasificar_documento(ruta)
    estado['ultimo_analisis'] = f"[documento detectado: {clase} · confianza {confianza:.0%}]"

    if clase == esperado:                                   # documento correcto
        estado['docs_ok'].append(esperado)
        estado['docs_pendientes'].pop(0); estado['intentos'] = 0
        msg = f"✅ Recibí tu {NOMBRE_DOC[esperado]}."
    else:                                                   # documento incorrecto o dudoso
        estado['intentos'] += 1
        if clase == 'dudoso':
            msg = ("❓ No pude identificar el documento con seguridad. Envía otra foto con buena luz, "
                   "sin reflejos y con el documento completo.")
        elif clase == 'otro':
            msg = f"⚠️ Esta imagen no parece un documento. Necesito tu {NOMBRE_DOC[esperado]}."
        else:
            msg = f"⚠️ Esta imagen parece tu {NOMBRE_DOC[clase]}, pero en este paso necesito tu {NOMBRE_DOC[esperado]}."
        if estado['intentos'] >= MAX_INTENTOS:              # después de 3 intentos, lo revisa RH
            estado['docs_revision'].append(esperado)
            estado['docs_pendientes'].pop(0); estado['intentos'] = 0
            msg += f"\nNo te preocupes: recursos humanos revisará tu {NOMBRE_DOC[esperado]} en la entrevista."
        else:
            return msg

    if estado['docs_pendientes']:
        return msg + "\n" + pedir_documento(estado)
    estado['paso'] = 'dia'
    return (msg + f"\nYa tengo tus documentos. Vamos a agendar tu entrevista {ENTREVISTA['modalidad']} "
            f"({ENTREVISTA['duracion']}). ¿Qué día te queda mejor? ({ENTREVISTA['dias']})")

In [ ]:
# ---------- Prefiltro ----------
def no_compatible(motivo, sugerir=False):
    t = f"Gracias por tus respuestas. Por ahora no cumples con un requisito: {motivo}."
    if sugerir: t += f"\nTe pueden interesar puestos que no piden experiencia: {puestos_sin_experiencia()}."
    return t + "\nSi tienes otra duda, con gusto te ayudo."

def compatible(estado):
    p = PUESTOS[estado['puesto']]
    estado['paso'] = 'documentos'
    estado['docs_pendientes'] = ['ine'] + (['licencia'] if p['requiere_licencia'] else []) + ['comprobante']
    estado['docs_ok'], estado['docs_revision'], estado['intentos'] = [], [], 0
    return (f"✅ ¡Tu perfil es compatible con el puesto de {p['nombre']}!\n"
            "Antes de agendar tu entrevista necesito tus documentos.\n" + pedir_documento(estado))

def despues_de_horario(estado):
    p = PUESTOS[estado['puesto']]
    if p['experiencia'] > 0:
        estado['paso'] = 'experiencia'
        return f"¿Cuántos años de experiencia tienes como {p['nombre']}? (escribe un número)"
    return despues_de_experiencia(estado)

def despues_de_experiencia(estado):
    if PUESTOS[estado['puesto']]['requiere_licencia']:
        estado['paso'] = 'licencia'
        return "¿Cuentas con licencia de conducir vigente? (sí / no)"
    return compatible(estado)

def iniciar_prefiltro(estado, puesto):
    estado['paso'] = 'puesto'
    if puesto: return procesar_paso(estado, [], puesto, '', None)
    return ("¡Qué gusto que quieras unirte! Te haré unas preguntas rápidas "
            "(escribe \"cancelar\" en cualquier momento).\n¿Qué puesto te interesa?\n" + lista_puestos())

def procesar_paso(estado, tokens, puesto, texto, archivo):
    paso = estado['paso']
    if paso == 'puesto':
        if not puesto: return "No reconocí el puesto. Elige uno de estos:\n" + lista_puestos()
        estado['puesto'], estado['paso'] = puesto, 'edad'
        return f"Perfecto, {PUESTOS[puesto]['nombre']}. ¿Cuántos años tienes? (escribe un número)"
    if paso == 'edad':
        edad = extraer_numero(tokens)
        if edad is None or not 10 <= edad <= 99: return "Escribe tu edad con un número, por ejemplo: 25"
        if edad < GENERAL['edad_minima']:
            estado['paso'] = None; return no_compatible(f"tener al menos {GENERAL['edad_minima']} años")
        p = PUESTOS[estado['puesto']]; estado['paso'] = 'horario'
        return f"Los horarios de {p['nombre']} son: {' / '.join(p['horarios'])}. ¿Puedes cubrir alguno? (sí / no)"
    if paso == 'horario':
        r = respuesta_si_no(texto)
        if r is None: return "Por favor responde sí o no: ¿puedes cubrir alguno de los horarios?"
        if not r: estado['paso'] = None; return no_compatible("disponibilidad para los horarios del puesto")
        return despues_de_horario(estado)
    if paso == 'experiencia':
        anios = extraer_numero(tokens)
        if anios is None: return "Escribe tus años de experiencia con un número, por ejemplo: 2"
        p = PUESTOS[estado['puesto']]
        if anios < p['experiencia']: estado['paso'] = None; return no_compatible(texto_experiencia(p), True)
        return despues_de_experiencia(estado)
    if paso == 'licencia':
        r = respuesta_si_no(texto)
        if r is None: return "Por favor responde sí o no: ¿tienes licencia de conducir vigente?"
        if not r: estado['paso'] = None; return no_compatible("licencia de conducir vigente")
        return compatible(estado)
    if paso == 'documentos':
        if archivo is None: return pedir_documento(estado) + " (o escribe \"cancelar\")"
        return procesar_documento(estado, archivo)
    if paso == 'dia':
        dia = extraer_dia(texto)
        if dia == 'fin_de_semana': return f"Las entrevistas son solo de {ENTREVISTA['dias']}. ¿Qué día de esos prefieres?"
        if dia is None: return "No reconocí el día. Escribe uno de estos: " + ', '.join(d.capitalize() for d in DIAS_HABILES) + "."
        if not espacios_libres(dia): return f"El {dia} ya no tiene espacios. ¿Te queda otro día?"
        estado['dia'], estado['paso'] = dia, 'hora'
        return f"Horarios disponibles el {dia}: {', '.join(espacios_libres(dia))}. ¿Cuál prefieres?"
    if paso == 'hora':
        hora, dia = extraer_hora(texto), estado['dia']
        if hora is None or hora not in espacios_del_dia(): return "Elige uno de estos horarios: " + ', '.join(espacios_libres(dia))
        if hora in AGENDA[dia]: return "Ese horario ya está ocupado. Disponibles: " + ', '.join(espacios_libres(dia))
        estado['hora'], estado['paso'] = hora, 'nombre'
        return "¡Perfecto! Para registrar tu cita, ¿cuál es tu nombre?"
    if paso == 'nombre':
        nombre = estado['mensaje_original'].strip().title()
        if len(nombre) < 2: return "Por favor escribe tu nombre."
        AGENDA[estado['dia']][estado['hora']] = nombre
        estado['paso'] = None
        p = PUESTOS[estado['puesto']]
        docs = ', '.join(NOMBRE_DOC[d].split(' (')[0] for d in estado['docs_ok']) or 'ninguno'
        t = (f"📅 Listo, {nombre}. Tu entrevista para {p['nombre']} quedó agendada:\n"
             f"  • Día: {estado['dia']}\n  • Hora: {estado['hora']}\n  • Lugar: {ENTREVISTA['lugar']}\n"
             f"  • Documentos recibidos: {docs}")
        if estado['docs_revision']:
            t += "\n  • Por revisar en la entrevista: " + ', '.join(NOMBRE_DOC[d].split(' (')[0] for d in estado['docs_revision'])
        return t + "\n¡Te esperamos!"

In [ ]:
# ---------- Motor principal ----------
def nuevo_estado():
    return {'paso': None, 'puesto': None, 'ultimo_puesto': None, 'mensaje_original': '', 'ultimo_analisis': ''}

def responder(mensaje, estado, archivo=None):
    tokens = tokens_de(mensaje)
    texto = ' '.join(tokens)
    puesto = identificar_puesto(tokens)
    estado['mensaje_original'], estado['ultimo_analisis'] = mensaje, ''

    # a) Dentro del prefiltro: se sigue el flujo guiado
    if estado['paso']:
        if contiene(texto, 'cancelar'):
            estado['paso'] = None
            return "Proceso cancelado. " + MENU
        return procesar_paso(estado, tokens, puesto, texto, archivo)

    if archivo is not None:
        return "Gracias. Los documentos se reciben al completar el prefiltro: escribe \"me interesa\" para empezar."

    # b) Respuesta corta "sí" después de ofrecer postularse
    ofrecio = estado.pop('ofrecio_postular', None)
    if ofrecio and len(tokens) <= 2 and respuesta_si_no(texto):
        return iniciar_prefiltro(estado, ofrecio)

    # c) Componente textual: clasificador de intenciones
    intencion, confianza = clasificar_intencion(mensaje)
    estado['ultimo_analisis'] = f"[intención: {intencion} · confianza {confianza:.0%}]"

    if puesto:
        estado['ultimo_puesto'] = puesto
        contenido = [t for t in tokens if t not in STOPWORDS]
        if len(contenido) == 1:                  # solo escribió el puesto: «cocinera», «para mesero»
            p = PUESTOS[puesto]; estado['ofrecio_postular'] = puesto
            return (f"{p['nombre']}: sueldo {p['sueldo']}, horario {' / '.join(p['horarios'])}, "
                    f"{texto_experiencia(p)}. ¿Quieres saber algo más o postularte?")
    elif intencion in ('funciones', 'sueldo', 'horarios'):
        puesto = estado['ultimo_puesto']        # pregunta de seguimiento: «¿y cuánto pagan?»

    if intencion == 'postularse':
        return iniciar_prefiltro(estado, puesto)
    return responder_faq(intencion, puesto)

## 6. Pruebas de integración
Cada conversación muestra el mensaje del usuario, el **análisis** de cada componente
(intención detectada por el SVM o documento detectado por la CNN) y la respuesta del chatbot.

> Coloca en `ChatbotRH/pruebas` fotos **nuevas**, que no se usaron para entrenar, con nombres que empiecen por
> `ine`, `licencia`, `comprobante` y `otro` (por ejemplo `ine_prueba.jpg`, `comprobante_mayo.pdf`).

In [ ]:
def archivo(prefijo):
    """Busca en la carpeta de pruebas el primer archivo cuyo nombre empiece con el prefijo."""
    for f in sorted(os.listdir(RUTA_PRUEBAS)):
        if f.lower().startswith(prefijo):
            return os.path.join(RUTA_PRUEBAS, f)
    raise FileNotFoundError(f"No hay un archivo que empiece con '{prefijo}' en {RUTA_PRUEBAS}")

def clase_detectada(estado):
    m = re.search(r'documento detectado: (\w+)', estado['ultimo_analisis'])
    return m.group(1) if m else 'dudoso'

def probar_conversacion(numero, titulo, mensajes):
    print('=' * 80); print(f"Conversación {numero}: {titulo}"); print('=' * 80)
    estado = nuevo_estado()
    for m in mensajes:
        if isinstance(m, tuple):                       # ('archivo', ruta): el usuario envía un documento
            ruta = m[1]
            print(f"Usuario: [envía archivo: {os.path.basename(ruta)}]")
            r = responder('', estado, archivo=ruta)
            mostrar_documento(ruta, clase_detectada(estado))
        else:
            print(f"Usuario: {m}")
            r = responder(m, estado)
        if MOSTRAR_ANALISIS and estado['ultimo_analisis']:
            print("   " + estado['ultimo_analisis'])
        print(f"Bot: {r}\n")

In [ ]:
probar_conversacion(1, 'Mensajes con faltas de ortografía y temas nuevos', [
    'holaa buenas tardes', 'q vacantes tienen', 'cuanto pagan d mesra',
    'dan seguro social?', 'quiero pedir unos tacos', 'grax'])

In [ ]:
probar_conversacion(2, 'Postulación completa con documentos correctos', [
    'me intereza el trabajo', 'mesera', '25', 'si', '2',
    ('archivo', archivo('ine')),
    ('archivo', archivo('comprobante')),
    'martes', '11:20', 'Laura Gómez'])

In [ ]:
probar_conversacion(3, 'Repartidor que envía primero un documento equivocado', [
    'quiero postularme de repartidor', '30', 'si', '3', 'si',
    ('archivo', archivo('comprobante')),     # el bot esperaba la INE
    ('archivo', archivo('ine')),
    ('archivo', archivo('licencia')),
    ('archivo', archivo('comprobante')),
    'jueves', '10', 'Carlos Ruiz'])

In [ ]:
probar_conversacion(4, 'Imagen que no es un documento', [
    'me interesa ser lavaloza', '19', 'si',
    ('archivo', archivo('otro')),
    ('archivo', archivo('ine')),
    'cancelar'])

In [ ]:
print("Entrevistas agendadas durante las pruebas:")
ver_agenda()

## 7. Modo interactivo
Escribe como candidato. Cuando el bot pida un documento, **presiona Enter sin escribir nada**
y aparecerá el botón para subir la foto o el PDF. Escribe `salir` para terminar.

In [ ]:
os.makedirs('/content/subidas', exist_ok=True)

def chatear():
    print("Asistente de RH - El Sauce (escribe 'salir' para terminar)\n")
    estado = nuevo_estado()
    while True:
        esperando_doc = estado['paso'] == 'documentos'
        mensaje = input("Tú (Enter para subir archivo): " if esperando_doc else "Tú: ")
        if normalizar(mensaje) in ('salir', 'exit') and not estado['paso']:
            print("Bot: ¡Hasta pronto!"); break
        ruta = None
        if esperando_doc and not mensaje.strip():
            subido = files.upload()
            if not subido: continue
            nombre, contenido = next(iter(subido.items()))
            ruta = os.path.join('/content/subidas', nombre)
            with open(ruta, 'wb') as f: f.write(contenido)
            print(f"Tú: [envía archivo: {nombre}]")
        r = responder(mensaje, estado, archivo=ruta)
        if ruta: mostrar_documento(ruta, clase_detectada(estado))
        if MOSTRAR_ANALISIS and estado['ultimo_analisis']: print("   " + estado['ultimo_analisis'])
        print("Bot:", r, "\n")

In [ ]:
chatear()